# Practical 1: Fast-Iteration CLI & Churn Pipeline

**Course:** Big Data Technologies (Day 1)  
**Starter Code:** `day1-cli-churn-pipeline.ipynb`  
**Submission Format:** STEMLearn Quiz & Executed Notebook PDF Evidence Pack  
**Execution Platform:** Google Colab (Required)  

---

### Execution Platform & Rationale
This practical assignment is required to be executed in **Google Colab** by uploading this notebook (`day1-cli-churn-pipeline.ipynb`).
* **Why Google Colab?** Colab provides a pristine, isolated Linux virtual machine with high-speed ephemeral NVMe disks and fast internet connectivity, without risking local environment corruption.

### Course GitLab Repository & Feature Branch Setup
Before commencing your work, configure your course repository on GitLab (`gitlab.com`):
1. Create a repository named `bdt-2026-<YOUR_STUDENT_NUMBER>` *(e.g., `bdt-2026-24681357`)*. Leave **Initialize repository with a README** checked so that the `main` branch is created.
2. Ensure **Visibility Level** is set strictly to **Private** (**Settings -> General -> Visibility**). Repositories set to Public or Internal will forfeit marks for Q15.
3. Navigate to **Project information -> Members** (or **Settings -> Members**) and invite **`jackdotwa`** with the **Developer** role so that the code may be reviewed.
4. Clone your repository locally and immediately create and switch to a feature branch:
   ```bash
   git checkout -b feat/churn-pipeline
   ```

**Managing Your Git Submissions:**
* All development and commits must occur on your `feat/churn-pipeline` branch.
* When completed, you will push this branch and open a Merge Request targeting `main`, assigning `jackdotwa` as the official Reviewer.

### Objective
Production transactional logs frequently reach tens or hundreds of gigabytes. Attempting to scan unindexed CSV dumps directly into in-memory DataFrames introduces severe query latency and memory bloat. In this practical assignment, you will:
1. **Triage & Profile** raw tables using fast Unix streams.
2. **Extract an isolated 1% relational slice** directly on disk with constant memory (< 10 MB RAM in < 20 s).
3. **Implement a feature extraction function** in Pandas adhering strictly to temporal cutoff rules.
4. **Train and evaluate a baseline churn model on a subset of data**.
5. **Version-control your work** on a feature branch, open an audited Merge Request, and export an executed PDF evidence pack.


## 0. Data Generation (Student Seeded)

Set your unique student number to seed the synthetic H&M data generator. This deterministically generates 50,000 customers, and thousands of articles and millions of transactions scaled to your student number (~1.6 GB, ~17,000,000 transactions).


In [ ]:
import os
import sys

# 1. Enter your unique 8-digit student number (used as the deterministic data generator seed)
STUDENT_ID = None  # TODO: Replace with your actual 8-digit student number (e.g. 24681357)
assert STUDENT_ID is not None, "❌ ACTION REQUIRED: Set STUDENT_ID to your unique 8-digit student number before running!"

# 2. Pre-flight check: Verify generate_hm_data.py is present; auto-fetch from course repo if missing
if not os.path.exists("generate_hm_data.py"):
    print("ℹ \x27generate_hm_data.py\x27 not found locally. Auto-fetching from course repository...")
    !wget -q -O generate_hm_data.py https://raw.githubusercontent.com/jackdotwa/us-ie-big-data-technologies/v2026.1.0-day1/intro-python-bash/generate_hm_data.py

if not os.path.exists("generate_hm_data.py") or os.path.getsize("generate_hm_data.py") == 0:
    raise FileNotFoundError(
        "\n====================================================================================\n"
        "❌ MISSING FILE: \x27generate_hm_data.py\x27 could not be found or downloaded!\n"
        "👉 ACTION REQUIRED: Upload \x27generate_hm_data.py\x27 to Google Colab using the Files\n"
        "   sidebar (folder icon on the left panel) before running this data generation cell.\n"
        "===================================================================================="
    )

# 3. Install GNU time benchmarking utility
!apt-get install -y -qq time

# 4. Generate the student-seeded dataset (~1.6 GB, ~17M rows)
!python generate_hm_data.py --student_id $STUDENT_ID


## Phase 1: Command Line Data Profiling (Bash)

Execute the raw exploration commands below directly in this notebook using IPython cell magics (`!` prefix) so that all steps are captured in your executed notebook:
* **Line counts:** Inspect total line counts of all three CSV files (`customers.csv`, `articles.csv`, `transactions_train.csv`).
* **Headers & Schema:** Inspect headers to identify the relational foreign keys.
* **Temporal boundaries:** Find the earliest and latest transaction dates in `transactions_train.csv`.

In [ ]:
# 1.1 Inspect line counts of customers.csv, articles.csv, transactions_train.csv
# TODO: Write your bash command here (e.g. !echo "hello world")


In [ ]:
# 1.2 Inspect headers to identify customer and article foreign keys
# TODO: Write your bash command here


In [ ]:
# 1.3 Find the earliest and latest transaction date in transactions_train.csv
# TODO: Write your bash command here


## Phase 2: Relational Stream Slicing (Bash)

### Constraints & Requirements:
1. **Customer Subsample:** Extract an exact 1% systematic sample of customers from `customers.csv` into `sample_customer_ids.txt`. Sample every 100th line starting from line 1 (the header): i.e. CSV lines 1, 101, 201, ... up to 50,001 using stream filtering tools (e.g. `sed` or `awk`). Your output file must contain exactly 501 lines (the header `customer_id` plus 500 customer hex IDs).
2. **Verification Hash:** Compute the SHA-256 hash of `sample_customer_ids.txt` (first 8 hex characters).
3. **Relational Stream Filter:** Filter `transactions_train.csv` (~17,000,000 rows, ~1.6 GB) into `transactions_sample.csv` containing only rows matching your sampled customer IDs.
   * **Target Performance:** Should execute in < 20 seconds (or < 5 s with awk) and consume < 10 MB of RAM.
   * **Requirement:** Must retain the CSV header row.

You can use the provided `benchmark_bash()` function below to benchmark your command's execution time and memory footprint directly!


In [ ]:
import subprocess
import time
import os
import shutil

def benchmark_bash(cmd: str):
    """
    Executes a bash command, measures wall-clock time and peak resident memory (RSS).
    Uses GNU /usr/bin/time for kernel-level peak RSS measurement.
    """
    gnu_time = '/usr/bin/time' if os.path.exists('/usr/bin/time') and os.access('/usr/bin/time', os.X_OK) else None
    if gnu_time:
        res = subprocess.run(
            [gnu_time, '-f', 'BENCH_RESULT: %e sec | %M KB', 'bash', '-c', cmd],
            capture_output=True, text=True
        )
        if res.returncode != 0:
            print(f"❌ Command exited with non-zero status ({res.returncode}):")
            print(res.stderr.strip())
            return {'seconds': 0.0, 'memory_mb': 0.0, 'success': False}
        lines = res.stderr.splitlines()
        bench_line = [l for l in lines if 'BENCH_RESULT:' in l]
        if bench_line:
            parts = bench_line[0].replace('BENCH_RESULT:', '').strip().split('|')
            sec = float(parts[0].replace('sec', '').strip())
            kb = float(parts[1].replace('KB', '').strip())
            mb = kb / 1024.0
            print(f"⏱️  Execution Time : {sec:.3f} seconds (Target: < 20.0 s)")
            print(f"💾 Peak Memory    : {mb:.2f} MB (Target: < 10.0 MB)")
            if sec < 20.0 and mb < 10.0:
                print("✅ Performance Constraints Satisfied!")
            else:
                print("⚠️ Performance constraint exceeded (optimize your pipeline).")
            return {'seconds': sec, 'memory_mb': mb, 'success': True}

    print("⚠️  Notice: GNU /usr/bin/time not found. Wall-clock measured, but peak RSS requires: !apt-get install -y time")
    t0 = time.perf_counter()
    res = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True)
    elapsed = time.perf_counter() - t0
    if res.returncode != 0:
        print(f"❌ Command exited with non-zero status ({res.returncode}):")
        print(res.stderr.strip())
        return {'seconds': 0.0, 'memory_mb': 0.0, 'success': False}
    print(f"⏱️  Execution Time : {elapsed:.3f} seconds (Target: < 20.0 s)")
    return {'seconds': elapsed, 'memory_mb': 0.0, 'success': True}


In [ ]:
# 2.1 Extract 1% customer sample (lines 1, 101, 201, ... 50001; exactly 501 lines) into sample_customer_ids.txt
# TODO: Write your bash command here


In [ ]:
# 2.2 Compute first 8 hex characters of the SHA-256 hash of sample_customer_ids.txt
# TODO: Write your bash command here


In [ ]:
# 2.3 Stream-filter transactions_train.csv into transactions_sample.csv and benchmark it
# Tip: grep -F -f runs in ~17 s (< 10 MB RAM).
# Pro-tip: For ultra-fast execution (< 5 s), load the 500 customer IDs into an awk associative array (using the two-file NR==FNR pattern) to filter field $2 with O(1) hash lookups.
filter_cmd = """
# TODO: Enter your high-performance bash filter pipeline here
""".strip()

if filter_cmd:
    benchmark_bash(filter_cmd)
else:
    print("Enter your bash pipeline in filter_cmd to benchmark.")


In [ ]:
# 2.4 Verify line count and total spend of sliced transactions
import pandas as pd
df_tx_sample = pd.read_csv('transactions_sample.csv')
print(f"Total Lines (including header): {len(df_tx_sample) + 1:,}")
print(f"Transactions Data Rows        : {len(df_tx_sample):,}")
print(f"Total Gross Revenue ($)       : {df_tx_sample['price'].sum():.2f}")


## Phase 3: Feature Engineering Function (Python)

### Business Specification:
Predict whether a customer will **churn** in the 62-day observation window (`2020-07-01` to `2020-08-31`).

**Rules & Specifications:**
* **Strict Temporal Demarcation:** Features must be derived **strictly** from transactions prior to `2020-07-01`.
* **Target Definition:** $y = 1$ (churned) if the customer made zero transactions between `2020-07-01` and `2020-08-31`; else $y = 0$.
* **Completeness Constraint:** All 500 customers in your sample must be present in the final matrix. Customers with zero purchases prior to cutoff must be imputed (`frequency = 0`, `monetary = 0.0`, `recency = 182.0`).
* **Missing Demographic Values:** Impute missing demographic age with the sample median.

In [ ]:
import pandas as pd
import numpy as np

def build_feature_store(customers_file, sample_ids_file, sample_tx_file, cutoff_date='2020-07-01'):
    """
    Builds customer-level feature matrix and churn target.
    
    Returns:
        pd.DataFrame with columns: ['customer_id', 'age', 'recency', 'frequency', 'monetary', 'churn']
    """
    # 1. Load data
    sample_ids = pd.read_csv(sample_ids_file)['customer_id'].astype(str)
    tx = pd.read_csv(sample_tx_file)
    cust = pd.read_csv(customers_file)
    
    # Format dates and string IDs
    tx['t_dat'] = pd.to_datetime(tx['t_dat'])
    tx['customer_id'] = tx['customer_id'].astype(str)
    cust['customer_id'] = cust['customer_id'].astype(str)
    cutoff = pd.to_datetime(cutoff_date)
    
    # 2. Extract pre-cutoff features (t_dat < cutoff_date)
    pre_tx = tx[tx['t_dat'] < cutoff]
    # TODO: Group pre_tx by customer_id and aggregate:
    #       - frequency: transaction count
    #       - monetary: total price sum
    #       - recency: days from latest purchase to cutoff: (cutoff - max_date).dt.days
    
    # 3. Compute post-cutoff churn target (t_dat >= cutoff_date)
    post_tx = tx[tx['t_dat'] >= cutoff]
    # TODO: y = 1 if customer made 0 purchases in post-cutoff window, else y = 0
    
    # 4. Merge all 500 customers, impute missing values:
    # TODO: Outer join with sample_ids and cust demographics
    #       - Impute missing demographic age with sample median
    #       - Impute customers with 0 pre-cutoff purchases: frequency=0, monetary=0.0, recency=182.0
    #       - Ensure output DataFrame contains all 500 customers with columns:
    #         ['customer_id', 'age', 'recency', 'frequency', 'monetary', 'churn']
    
    return None


In [ ]:
# Execute and inspect feature store
df_feat = build_feature_store('customers.csv', 'sample_customer_ids.txt', 'transactions_sample.csv')

if df_feat is not None:
    df_feat.info()
    display(df_feat.head(10))
    print(f"Mean Customer Age : {df_feat['age'].mean():.2f}")
    print(f"Std Dev Age       : {df_feat['age'].std():.2f}")
    print(f"Mean Recency (days): {df_feat['recency'].mean():.1f}")
    print(f"Churn Percentage  : {(df_feat['churn'].mean() * 100):.1f}%")
else:
    print("Implement build_feature_store() first.")


### Phase 3 Milestone Commit
Once your `build_feature_store(...)` function executes cleanly and matches the completeness constraints, stage and commit your progress on your feature branch (`feat/churn-pipeline`):
```bash
git add day1/day1-cli-churn-pipeline.ipynb
git commit -m "feat(features): implement build_feature_store with cutoff and completeness rules"
```


In [ ]:
# Note: Execute git commits in your local workstation terminal inside your cloned course repo.
print("Phase 3 complete - remember to commit locally!")


## Phase 4: Baseline Model Prototyping (Python)

Evaluate your feature matrix with the baseline modeling harness below.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def train_and_eval_baseline(df_features):
    X = df_features[['age', 'recency', 'frequency', 'monetary']]
    y = df_features['churn']
    
    # Stratified 80/20 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    model = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    top_feat = X.columns[model.feature_importances_.argmax()]
    test_churns = int((y_test == 1).sum())
    
    print(f"Validation Test ROC-AUC  : {auc:.3f}")
    print(f"Top Predictive Feature   : {top_feat}")
    print(f"Test Churn Cases (y == 1): {test_churns} / {len(y_test)}")
    return model, auc

if df_feat is not None:
    model, auc = train_and_eval_baseline(df_feat)


## Phase 5: Version Control & Evidence Submission

### Tasks:
1. In your local repository clone on branch `feat/churn-pipeline`, configure `.gitignore` to prevent tracking raw data:
   ```bash
   echo "sample_customer_ids.txt" >> .gitignore
   echo "*.csv" >> .gitignore
   echo ".ipynb_checkpoints/" >> .gitignore
   ```
2. Stage and commit your final notebook on branch `feat/churn-pipeline`:
   ```bash
   git add .gitignore day1/day1-cli-churn-pipeline.ipynb
   git commit -m "feat(day1): complete fast-iteration cli and churn pipeline"
   ```
3. Push your feature branch to your private GitLab repository:
   ```bash
   git push -u origin feat/churn-pipeline
   ```
4. **Open a GitLab Merge Request:**
   * Navigate to your project on GitLab and click **Merge requests -> New merge request**.
   * **Source branch:** `feat/churn-pipeline` | **Target branch:** `main`.
   * **Title:** `feat(day1): complete fast-iteration cli and churn pipeline`.
   * **Reviewer:** Assign **`jackdotwa`** as the Reviewer.
   * Click **Create merge request** and leave it **OPEN** for evaluation.
5. **Export Notebook Evidence PDF:**
   * In Google Colab, select **File -> Print -> Save as PDF** (or use `nbconvert`).
   * Ensure all cell execution traces, benchmark timings, and model evaluations are fully visible, and save as `day1-cli-churn-pipeline.pdf`.

### STEMLearn Submission Deliverables:
* **Q15 (5.0 pts):** Student GitLab Repository URL (Audit will verify **Private** visibility and Developer access).
* **Q16 (5.0 pts):** GitLab Merge Request URL (Audit will verify open MR on `feat/churn-pipeline` with Reviewer `jackdotwa`).
* **S02 (0.0 pts):** Upload `day1-cli-churn-pipeline.pdf` (Mandatory audit prerequisite).


In [ ]:
# Local Git Verification Reminder:
# Ensure your local clone has .gitignore, your notebook, and clean git status (no data files tracked).
# Before opening your Merge Request, check locally: git log -n 5 --oneline && git status
print("Ready for local commit and STEMLearn submission!")
